# 2. Fine-tuning de NLLB avec LoRA sur notre corpus

**Objectif** : adapter NLLB-200-distilled-600M a NOTRE corpus (ewe 1913 +
segond 1910 + NLLB filtre) avec **LoRA** (Low-Rank Adaptation), pour depasser
la baseline.

## Pourquoi LoRA et pas un fine-tuning complet ?

- Un fine-tuning complet modifierait les **600M de parametres** : GPU sature,
  heures d'entrainement, risque d'oubli catastrophique.
- **LoRA** ne modifie que de petits "adaptateurs" (~0,5 % des parametres)
  ajoutes aux couches d'attention : rapide, leger, et le modele de base reste
  intact.
- Resultat : ~20-40 min d'entrainement sur un T4 gratuit pour 3 epoques.

## Pipeline

1. Charger `train.tsv` (52 512 paires) et `dev.tsv` (6 564 paires) depuis GitHub
2. Tokeniser les paires (langue source + langue cible NLLB)
3. Ajouter les adaptateurs LoRA
4. Entrainer avec `Seq2SeqTrainer` (HuggingFace)
5. Evaluer sur `test.tsv` et **comparer avec la baseline** (notebook 1)

In [1]:
# Installation (peft = bibliotheque officielle LoRA de HuggingFace)
!pip install -q transformers sacrebleu pandas sentencepiece datasets peft accelerate

print("Dependances installees")

Dependances installees


In [2]:
# Diagnostic GPU
# Colab fournit un GPU (T4) gratuitement, mais il faut l'activer :
#   menu Executer > Changer le type d'execution > T4 GPU
#   puis Executer > Redemarrer la session (obligatoire).
import torch

print("CUDA disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    print("Memoire :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "Go")
else:
    print("Attention : execution sur CPU (lent). Active le GPU T4 puis redemarre la session.")
    print("Si Colab ne propose pas de GPU (quota), utilise Kaggle : Accelerator > GPU T4.")

CUDA disponible : True
GPU : Tesla T4
Memoire : 15.6 Go


In [3]:
# Imports
import torch
import pandas as pd
from sacrebleu.metrics import CHRF, BLEU
import numpy as np
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments,
                          DataCollatorForSeq2Seq)
from datasets import Dataset
from peft import LoraConfig, get_peft_model

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device utilise :", device)

Device utilise : cuda


In [4]:
# Chargement train / dev / test depuis le repo GitHub public
BASE = "https://raw.githubusercontent.com/cherif-tg/tg_nlp_toolkit/main/data/processed/v0.3/"

def charger(nom):
    return pd.read_csv(BASE + nom, sep="\t", on_bad_lines="skip")

train = charger("train.tsv")
dev   = charger("dev.tsv")
test  = charger("test.tsv")
print("train =", len(train), "| dev =", len(dev), "| test =", len(test))
print(train.head(2))

train = 47610 | dev = 5886 | test = 5955
  source                                                 fr  \
0  bible              Éternel, mon Dieu! Si j’ai fait cela,   
1   nllb  Il a dit: "Pourquoi m'as-tu fait sortir de mon...   

                                                 ewe  
0  Yehowa, nye Mawu, ne mewo nusia, nu tovo 1e de...  
1  Bo madzo madzo nye nye ya yi dzi, abe: "Demiat...  


In [5]:
# Chargement du modele + tokenizer
MODEL_NAME = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

# On gelee le modele de base : seuls les adaptateurs LoRA seront entrainables.
model.config.use_cache = False  # requis par le Trainer pendant l'entrainement
print("Modele charge")

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Modele charge


In [6]:
# Preparation des donnees au format attendu par le Trainer
# Chaque exemple : "input_ids" = phrase source tokenisee (langue source),
#                  "labels"   = phrase cible tokenisee (langue cible).
# Le tokenizer NLLB encode la langue via src_lang et forced_bos_token_id.

# Modifie la fonction pour tokeniser un seul exemple, sans padding ou retour de tensors,
# pour que le DataCollator puisse gerer le batching et le padding de maniere appropriee.
def process_example(source_text, target_text):
    tokenizer.src_lang = "fra_Latn"
    # Tokenize source text, get input_ids and attention_mask as lists
    source_tokenized = tokenizer(source_text, truncation=True, max_length=128)

    tokenizer.src_lang = "ewe_Latn"
    # Tokenize target text, get input_ids (which will be our labels) as a list
    target_tokenized = tokenizer(target_text, truncation=True, max_length=128)

    return {
        "input_ids": source_tokenized["input_ids"],
        "attention_mask": source_tokenized["attention_mask"],
        "labels": target_tokenized["input_ids"], # Labels are the tokenized target sequence
    }

train_ds = Dataset.from_list(
    [process_example(str(fr), str(ee)) for fr, ee in zip(train["fr"], train["ewe"])]
)
dev_ds = Dataset.from_list(
    [process_example(str(fr), str(ee)) for fr, ee in zip(dev["fr"], dev["ewe"])]
)
print("Datasets prets : train", len(train_ds), "| dev", len(dev_ds))
print("Exemple de cles :", list(train_ds[0].keys()))

Datasets prets : train 47610 | dev 5886
Exemple de cles : ['input_ids', 'attention_mask', 'labels']


In [7]:
import torch

# Configuration LoRA
# On ajoute des adaptateurs sur les projections Q et V de l'attention
# (cible classique pour les modeles seq2seq).

# Fix for ImportError: Incompatible torchao version.
# This issue typically arises from a mismatch between the PEFT library
# and its underlying dependency, torchao. Upgrading torchao and peft can resolve this.
# Note: After running this cell, you will likely need to restart the Colab runtime
# (Runtime > Restart runtime) for the changes to take effect.
#!pip install torchao --upgrade
#!pip install peft --upgrade

lora_config = LoraConfig(
    r=16,                 # rang de la factorisation (plus = plus de capacite)
    lora_alpha=32,        # echelle de mise a jour (souvent 2 x r)
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,    # regularisation
    bias="none",
    task_type="SEQ_2_SEQ_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Attendu : ~0,5 % des parametres entrainables seulement !

trainable params: 2,359,296 || all params: 617,433,088 || trainable%: 0.3821


In [8]:
# Metrique d'evaluation pendant l'entrainement : chrF++ sur le dev set
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    refs = [[r] for r in decoded_labels]
    chrf = CHRF().corpus_score(decoded_preds, refs)
    return {"chrF++": chrf.score}

collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)
print("Metrique + collator prets")

Metrique + collator prets


In [9]:
# Configuration de l'entrainement (adaptee a un T4 gratuit)
training_args = Seq2SeqTrainingArguments(
    output_dir="nllb-ewe-lora",
    num_train_epochs=3,             # 3 passages sur le corpus
    per_device_train_batch_size=8,  # 8 paires par lot (T4 ~ 16 Go)
    per_device_eval_batch_size=8,
    learning_rate=3e-4,
    warmup_steps=200,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",          # evaluation a chaque fin d'epoque
    save_strategy="epoch",
    predict_with_generate=True,     # genere de vraies traductions pour la metrique
    generation_max_length=128,
    fp16=True,                      # demi-precision : plus rapide sur T4
    report_to="none",
    push_to_hub=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

print("Trainer pret - lance l'entrainement avec la cellule suivante")

Trainer pret - lance l'entrainement avec la cellule suivante


In [10]:
# LANCEMENT DE L'ENTRAINEMENT (~20-40 min sur T4)
trainer.train()

print("Entrainement termine !")

Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Chrf++
1,2.600699,2.317578,39.505953
2,2.345280,2.223955,41.215892
3,2.449731,2.195240,55.537243


Entrainement termine !


In [11]:
# Evaluation finale sur le TEST set (jamais vu par le modele)
def traduire_model(textes, tgt="ewe_Latn", max_len=128, batch_size=16):
    tokenizer.src_lang = "fra_Latn"
    resultats = []
    for i in range(0, len(textes), batch_size):
        lot = textes[i:i + batch_size]
        enc = tokenizer(lot, return_tensors="pt", padding=True,
                        truncation=True, max_length=max_len).to(device)
        with torch.no_grad():
            gen = model.generate(
                **enc,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt),
                max_new_tokens=max_len,
                num_beams=4,
            )
        resultats += tokenizer.batch_decode(gen, skip_special_tokens=True)
    return resultats

preds = traduire_model([str(x) for x in test["fr"].tolist()])
refs = [str(x) for x in test["ewe"].tolist()]
chrf = CHRF().corpus_score(preds, [refs])
bleu = BLEU().corpus_score(preds, [refs])

print("FR -> EWE apres fine-tuning LoRA")
print("   chrF++ :", round(chrf.score, 2), " (a comparer avec la baseline)")
print("   BLEU   :", round(bleu.score, 2))

for i in range(3):
    print("--- Exemple", i + 1, "---")
    print("FR :", test['fr'].iloc[i])
    print("Ref:", refs[i])
    print("Pred:", preds[i])

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

FR -> EWE apres fine-tuning LoRA
   chrF++ : 41.83  (a comparer avec la baseline)
   BLEU   : 18.71
--- Exemple 1 ---
FR : Ils ont regardé, tout stupéfaits,
Ref: Esi wokpoe la, wofe nu ku, dzidzi fo wo, eye wosi
Pred: Wo- kpoe, eye wofe mo wu wo, eye wofe mo wu wo
--- Exemple 2 ---
FR : C'est une drôle de question, non ?
Ref: Ðe biabia sia mele vevie ŋutɔ oa?
Pred: Ðe biabia sia mewɔ nuku ŋutɔ oa?
--- Exemple 3 ---
FR : Les Juifs enterraient leurs morts tout de suite après leur décès, en général dans la journée même.
Ref: Yudatɔwo ɖia woƒe ame kukuwo kaba, zi geɖe le ŋkeke si dzi amea ku le.
Pred: Yudatɔwo ɖia woƒe ame kukuwo enumake le woƒe ku megbe, zi geɖe le ŋkeke ma ke dzi.


In [ ]:
# Evaluation EWE -> FR avec le modele fine-tune (comparaison complete)
def traduire_model_inverse(textes, tgt="fra_Latn", max_len=128, batch_size=16):
    tokenizer.src_lang = "ewe_Latn"
    resultats = []
    for i in range(0, len(textes), batch_size):
        lot = textes[i:i + batch_size]
        enc = tokenizer(lot, return_tensors="pt", padding=True,
                        truncation=True, max_length=max_len).to(device)
        with torch.no_grad():
            gen = model.generate(
                **enc,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt),
                max_new_tokens=max_len,
                num_beams=4,
            )
        resultats += tokenizer.batch_decode(gen, skip_special_tokens=True)
    return resultats

preds_ee_fr = traduire_model_inverse([str(x) for x in test["ewe"].tolist()])
refs_fr = [str(x) for x in test["fr"].tolist()]
chrf_ee_fr = CHRF().corpus_score(preds_ee_fr, [refs_fr])
bleu_ee_fr = BLEU().corpus_score(preds_ee_fr, [refs_fr])

print("EWE -> FR apres fine-tuning LoRA")
print("   chrF++ :", round(chrf_ee_fr.score, 2), " (a comparer avec la baseline)")
print("   BLEU   :", round(bleu_ee_fr.score, 2))


In [12]:
# Sauvegarde du modele + export vers HuggingFace (optionnel)
# 1) Sauvegarde locale (dossier modele complet)
model.save_pretrained("nllb-ewe-lora-final")
tokenizer.save_pretrained("nllb-ewe-lora-final")
print("Modele sauvegarde dans nllb-ewe-lora-final/")

# 2) Export vers ton compte HuggingFace (cheriftenga)
# Decommente et execute APRES t'etre connecte :
#   from huggingface_hub import notebook_login
#   notebook_login()   # colle ton token (Settings > Access Tokens)
#
#   model.push_to_hub("cheriftenga/nllb-200-distilled-600M-ewe-lora")
#   tokenizer.push_to_hub("cheriftenga/nllb-200-distilled-600M-ewe-lora")
print("Pret pour l'export (voir instructions commentees)")

Modele sauvegarde dans nllb-ewe-lora-final/
Pret pour l'export (voir instructions commentees)


## Lecture des resultats

- Si **chrF++ fine-tune > chrF++ baseline** (notebook 1) : notre corpus apporte
  un vrai gain -> le corpus v0.3 est **utile et publiable**.
- Si le gain est faible : verifier (a) le nombre d'epoques, (b) le `r` de LoRA,
  (c) la taille du corpus. Les donnees restent la contrainte principale en
  low-resource.

**Prochaines etapes** : demo Gradio (P3), publication HuggingFace,
puis traduction manuelle des grilles (10 themes) pour couvrir le domaine
sante/administration.